# MWE of distributions

In [17]:
import qmcpy as qp 
import scipy as sp
import numpy as np
from scipy import stats

## True Measure Way
It seems that we can only specify one kind of independent marginal

In [18]:
unif_sampler = qp.Sobol(1)
Gauss_sampler = qp.Gaussian(unif_sampler)
Gauss_sampler(8)

array([[ 0.2055682 ],
       [-1.09605579],
       [ 2.45907876],
       [-0.57786511],
       [ 0.45891349],
       [-1.24187295],
       [ 0.77195029],
       [-0.1423475 ]])

## SciPy Wrapper Way
Here we can stack different independent marginals, which we cannot do with `truemeasure` built-in

In [19]:
unif_sampler = qp.Sobol(2)

In [20]:
indep_marginals = [stats.uniform(-1.0, 4.0), stats.norm(0.0, 1.0)]
uniform_plus_Gauss_sampler = qp.SciPyWrapper(unif_sampler, indep_marginals)
uniform_plus_Gauss_sampler(8)

array([[ 0.91155445,  0.94883548],
       [ 2.38214814, -0.74899139],
       [-0.62244725, -0.40770336],
       [ 1.90686988,  0.59397776],
       [ 0.01005816, -1.39074617],
       [ 2.53950499,  2.08378418],
       [-0.45532597,  0.2206031 ],
       [ 1.01513802, -0.07460228]])

## Importance sampling
With `truemeasure` we can change the measure from uniform to something, but I do not see how we can chain variable transformations

In [29]:
integrand_vs_Gauss = qp.CustomFun(true_measure = Gauss_sampler, g = lambda x : x[:,0]**2)
stop_criterion = qp.CubQMCNetG(integrand_vs_Gauss, abs_tol=1e-6)
sol, data = stop_criterion.integrate()
print("sol =", sol)
print(data)

sol = 1.0000000939874085
Data (Data)
    solution        1.000
    comb_bound_low  1.000
    comb_bound_high 1.000
    comb_bound_diff 1.05e-06
    comb_flags      1
    n_total         2^(23)
    n               2^(23)
    time_integrate  1.156
CubQMCNetG (AbstractStoppingCriterion)
    abs_tol         1.00e-06
    rel_tol         0
    n_init          2^(10)
    n_limit         2^(35)
CustomFun (AbstractIntegrand)
Gaussian (AbstractTrueMeasure)
    mean            0
    covariance      1
    decomp_type     PCA
DigitalNetB2 (AbstractLDDiscreteDistribution)
    d               1
    replications    1
    randomize       LMS DS
    gen_mats_source joe_kuo.6.21201.txt
    order           RADICAL INVERSE
    t               63
    alpha           1
    n_limit         2^(32)
    entropy         91458597834614516290585705836369685321


### We try different tolerances and we see that sample size increases roughly like 1/tolerance

In [31]:
abs_tol_vec = [1e-3, 1e-4, 1e-5, 1e-6, 1e-7]
n_total_vec = np.zeros(len(abs_tol_vec), dtype=int)
for ii, abs_tol in enumerate(abs_tol_vec):
    _,data = qp.CubQMCNetG(integrand_vs_Gauss, abs_tol=abs_tol).integrate()
    n_total_vec[ii] = data.n_total

print("abs_tol_vec =", abs_tol_vec)
print("n_total_vec =", n_total_vec)
print("n_increase = ", n_total_vec/n_total_vec[0])

abs_tol_vec = [0.001, 0.0001, 1e-05, 1e-06, 1e-07]
n_total_vec = [     8192     16384    524288   8388608 134217728]
n_increase =  [1.0000e+00 2.0000e+00 6.4000e+01 1.0240e+03 1.6384e+04]


### So we might want to try importance sampling, say using a Gaussian with a different variance or Pareto, but I do not know how without rewriting the integrand